In [11]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Optional
from google.colab import files


In [ ]:
# Helper

def export_df_to_excel(
    df: pd.DataFrame,
    file_path: str,
    sheet_name: str = "data",
    index: bool = True,
):
    df.to_excel(
        file_path,
        sheet_name=sheet_name,
        index=index
    )


# 1 - Importer Indice historique


In [9]:
def load_excel_file():
    """
    Importe un fichier Excel depuis Colab et retourne :
    - le DataFrame complet
    - la colonne Date convertie en datetime
    - la colonne Index convertie en numérique
    """

    uploaded = files.upload()
    filename = next(iter(uploaded))

    df = pd.read_excel(filename)

    dates = pd.to_datetime(df["Date"])

    index = pd.to_numeric(
        df["Index"].astype(str).str.replace(",", "."),
        errors="coerce"
    )

    return df, dates, index

In [10]:
df1, dates1, index1 = load_excel_file()
# df2, dates2, index2 = load_excel_file()
# df3, dates3, index3 = load_excel_file()

print(df1.head())
#print(df2.head())

Saving Data_SAF.xlsx to Data_SAF (3).xlsx
        Date        Index
0 2005-12-29  1000.000000
1 2005-12-30   991.560128
2 2006-01-03   997.860255
3 2006-01-04   999.580579
4 2006-01-05  1000.846722


# 2 - Traitement de Time Serie


In [13]:
def load_index_series_df(df: pd.DataFrame) -> pd.Series:
    """ load_index_series_AB()
    Transforme un DataFrame contenant :
      - Colonne 0 : Date
      - Colonne 1 : Index level

    Retourne :
      pd.Series avec DatetimeIndex et valeurs float.
    """

    date_col = df.columns[0]
    level_col = df.columns[1]

    # Parse date
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    # Parse level (format français)
    lvl = (
        df[level_col]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    df[level_col] = pd.to_numeric(lvl, errors="coerce")

    s = (
        df[[date_col, level_col]]
        .dropna()
        .drop_duplicates(subset=date_col)
        .set_index(date_col)[level_col]
        .sort_index()
    )

    if s.empty:
        raise ValueError("Series vide après nettoyage.")

    if (s <= 0).any():
        raise ValueError("Certaines valeurs sont <= 0.")

    return s

In [16]:
levels = load_index_series_df(df1)
print(levels)

Date
2005-12-29    1000.000000
2005-12-30     991.560128
2006-01-03     997.860255
2006-01-04     999.580579
2006-01-05    1000.846722
                 ...     
2025-05-08    2621.333962
2025-05-09    2633.512877
2025-05-12    2715.116609
2025-05-13    2740.053076
2025-05-14    2748.187278
Name: Index, Length: 4974, dtype: float64


#3 - Black&Schole Model

In [19]:

@dataclass
class BSParams:
    r_annual_cc: float
    sigma_annual: float
    s0: float
    s0_date: Optional[pd.Timestamp]


def estimate_sigma_from_history(levels: pd.Series, day_count: int = 365) -> float:
    """sigma_annual = std(log-return daily) * sqrt(day_count)"""
    s = levels.sort_index()
    logrets = np.log(s / s.shift(1)).dropna()
    return float(logrets.std(ddof=1) * np.sqrt(day_count))


def get_s0(
    levels: pd.Series,
    start_date: str,
    default_s0: float = 1000.0,
    use_real_if_available: bool = False,
) -> tuple[float, Optional[pd.Timestamp]]:
    """Chọn S0: default hoặc lấy đúng level tại start_date nếu có."""
    if not use_real_if_available:
        return float(default_s0), None

    d = pd.Timestamp(start_date)
    if d in levels.index:
        return float(levels.loc[d]), d
    return float(default_s0), None


def build_bs_params_simple(
    levels: pd.Series,
    start_date: str,
    r_neutre_annual_cc: float,
    day_count: int = 365,
    s0_default: float = 1000.0,
    use_real_s0: bool = False,
) -> BSParams:
    """
    Build params đơn giản:
      - sigma: từ dữ liệu quá khứ
      - r_cc: = r_neutre (bạn truyền vào)
      - s0: default hoặc lấy level thật tại start_date nếu có
    """
    s0, s0_date = get_s0(
        levels,
        start_date=start_date,
        default_s0=s0_default,
        use_real_if_available=use_real_s0,
    )

    sigma = estimate_sigma_from_history(levels, day_count=day_count)

    return BSParams(
        r_annual_cc=float(r_neutre_annual_cc),
        sigma_annual=float(sigma),
        s0=float(s0),
        s0_date=s0_date,
    )

In [26]:
r_neutre_cc = 0.0691009521484375

params = build_bs_params_simple(
    levels=levels,
    start_date="2025-12-31",
    r_neutre_annual_cc=r_neutre_cc,
    s0_default=1000,
    use_real_s0=False,
)

print(params)

BSParams(r_annual_cc=0.0691009521484375, sigma_annual=0.2558122680177892, s0=1000.0, s0_date=None)


# 4 - Path GBM

In [22]:
def simulate_gbm_monthly(
    s0: float,
    r_annual_cc: float,
    sigma_annual: float,
    start_date: str,
    n_months: int,
    n_sims: int,
    seed: Optional[int] = 42,
) -> pd.DataFrame:
    """GBM monthly simulation, output scenario x dates."""
    rng = np.random.default_rng(seed)

    dt = 1.0 / 12.0
    dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")

    Z = rng.standard_normal(size=(n_months, n_sims))
    drift = (r_annual_cc - 0.5 * sigma_annual**2) * dt
    diffusion = sigma_annual * np.sqrt(dt) * Z

    log_paths = np.vstack([np.zeros((1, n_sims)), np.cumsum(drift + diffusion, axis=0)])
    paths = s0 * np.exp(log_paths)

    df = pd.DataFrame(paths.T, index=np.arange(1, n_sims + 1), columns=dates)
    df.index.name = "scenario"
    return df

In [25]:
    paths = simulate_gbm_monthly(
        s0=params.s0,
        r_annual_cc=params.r_annual_cc,
        sigma_annual=params.sigma_annual,
        start_date="2025-12-31",
        n_months=145,
        n_sims=10_000,
        seed=42,
    )
    print(paths)

          2025-12-31   2026-01-31   2026-02-28   2026-03-31   2026-04-30  \
scenario                                                                   
1             1000.0  1025.862869  1042.459994   925.114746   861.141640   
2             1000.0   928.887558   995.670813  1066.631773  1066.132287   
3             1000.0  1060.192027   953.858024   952.662903   865.120049   
4             1000.0  1075.181296  1055.962563  1013.270491   964.050429   
5             1000.0   868.449670   926.090136   911.855275   898.278655   
...              ...          ...          ...          ...          ...   
9996          1000.0  1130.512586  1291.790929  1317.124774  1225.493203   
9997          1000.0   998.367465   943.108105   959.355509   887.451229   
9998          1000.0  1009.370718  1061.536702  1222.259735  1372.273292   
9999          1000.0  1089.904775  1136.937003  1172.955889  1100.016991   
10000         1000.0   989.583530  1082.560470  1025.008575   999.259945   

           

/tmp/ipykernel_628/3503761269.py:14: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(start=pd.Timestamp(start_date), periods=n_months + 1, freq="M")


# 5 - decrement

In [27]:

from typing import Literal
DecrementMode = Literal["flat", "cum"]

def decrement_paths(
    df: pd.DataFrame,
    decrement_value: float = 50.0,   # 50 points per year
    periods_per_year: int = 12,      # monthly
    mode: DecrementMode = "cum",     # với logic bạn nêu: dùng "cum"
) -> pd.DataFrame:
    """
    Linear decrement per period with cumulative option.

    Your target behavior (mode="cum", monthly):
      col0 (t0): 0
      col1:  -50/12
      col2:  -2*50/12
      ...
      colk:  -k*50/12

    mode:
      - "flat": subtract 50/12 mỗi kỳ (không tích lũy)
      - "cum" : subtract k*(50/12) (tích lũy theo số kỳ)
    """
    df_dec = df.copy()

    # ensure datetime columns + sort
    cols = pd.to_datetime(df_dec.columns)
    order = np.argsort(cols.values)
    df_dec = df_dec.iloc[:, order]

    n_cols = df_dec.shape[1]
    if n_cols == 0:
        return df_dec

    per_period = float(decrement_value) / float(periods_per_year)  # 50/12

    # k = 0,1,2,...,n_cols-1 where col0 is t0
    k = np.arange(n_cols, dtype=float)

    if mode == "flat":
        decrements = per_period * (k > 0)          # col0=0, các cột sau trừ 50/12
    elif mode == "cum":
        decrements = per_period * k                # col0=0, col1=1*50/12, col2=2*50/12,...
    else:
        raise ValueError("mode must be 'flat' or 'cum'")

    df_dec.iloc[:, :] = df_dec.values - decrements
    return df_dec

In [28]:
path_decrement = decrement_paths(paths, decrement_value=50, mode = 'cum')

In [29]:
path_decrement.head()

,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31,2026-09-30,...,2037-04-30,2037-05-31,2037-06-30,2037-07-31,2037-08-31,2037-09-30,2037-10-31,2037-11-30,2037-12-31,2038-01-31
scenario,,,,,,,,,,,,,,,,,,,,,
1,1000.0,1021.696203,1034.126660,912.614746,844.474973,787.396035,833.095952,882.747050,766.884344,805.127221,...,1075.777219,936.918683,666.873096,655.707250,540.795634,580.067027,638.647134,644.587937,681.642614,731.405525
2,1000.0,924.720891,987.337480,1054.131773,1049.465620,953.655082,828.015882,806.995786,847.068254,889.294136,...,259.542262,110.958383,141.779543,164.361785,293.664158,274.851697,303.140505,247.839118,305.945257,284.455953
3,1000.0,1056.025361,945.524691,940.162903,848.453382,868.491560,851.377087,954.848710,977.602081,1009.232103,...,1125.126238,1099.468857,1239.884122,1105.557211,1100.726005,994.006986,1055.008679,1093.653297,1112.585885,1257.756984
4,1000.0,1071.014629,1047.629230,1000.770491,947.383763,957.156468,880.050892,813.432963,773.281387,879.198614,...,765.904285,612.716293,486.053586,427.088524,629.977585,637.604131,676.520777,769.216775,627.213774,529.163620
5,1000.0,864.283003,917.756802,899.355275,881.611988,743.677259,922.416480,810.198310,739.707322,716.042935,...,5218.182124,4780.917968,4744.292689,5111.755243,5055.068847,5306.201154,5234.918772,4487.499075,4491.506402,4469.133238
